# WestQuant Representation Stack — Combined QoolQit Notebook

**QoolQit version: 1.4.0** (recorded via `westquant_qoolqit.common.QOOLQIT_VERSION`)

This notebook demonstrates the full WestQuant Representation Stack for the
Pasqal QoolQit Contest, combining both projects:

- **Project A — WestQuant Representation Scheduler**: search over physical
  embeddings / hardware realizations for a fixed logical Hamiltonian.
- **Project B — WestQuant Hamiltonian Representation Explorer**: search over
  mathematically valid Hamiltonian representations of the same problem.

Central scientific thesis:

> A mathematical optimization problem does not necessarily determine a unique
> useful quantum representation. Representation itself can be treated as an
> optimization variable.

Pipeline: `P -> H_i -> R_ij -> Q_ijk -> Pareto selection`

In [ ]:
# Installation (run once)
%pip install -q "qoolqit==1.4.0"
%pip install -q "git+https://github.com/VesterlundCoder/QoolQit.git@qoolqit-contest-v1.0"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qoolqit
print('QoolQit version:', qoolqit.__version__)

from westquant_qoolqit.common import (
    BinaryQuadraticHamiltonian, solve_exact, QOOLQIT_VERSION, capture_environment,
)
from westquant_qoolqit.hamiltonian_explorer import HamiltonianRepresentationExplorer
from westquant_qoolqit.representation_scheduler import RepresentationScheduler, HalvingConfig
from westquant_qoolqit.combined import combined_search
from westquant_qoolqit.benchmarks import mwis_path

assert QOOLQIT_VERSION == qoolqit.__version__
env = capture_environment(seed=42)
print('Environment:', env.qoolqit_version, 'on Python', env.python_version)

## 1. Problem: Maximum Weighted Independent Set (MWIS)

Small instance so the exact classical optimum can be computed by exhaustive
enumeration and used as ground truth.

In [ ]:
h_problem, mwis_info = mwis_path(n=5, seed=42)
print('Weights:', mwis_info['weights'])
print('Edges:', mwis_info['edges'])
exact = solve_exact(h_problem)
print('Exact optimum energy:', exact.optimum_energy)
print('Exact optimum state(s):', exact.optimum_states.tolist())
print('Classical gap:', exact.gap)

## 2. Project B — Hamiltonian Representation Explorer

Generate multiple mathematically valid Hamiltonian representations and
verify their equivalence. Each is classified into exactly one of:
`EXACT_EQUIVALENT`, `GROUND_STATE_EQUIVALENT`,
`SAME_PROBLEM_DIFFERENT_DYNAMICS`, `APPROXIMATE`, `INVALID`.

In [ ]:
explorer = HamiltonianRepresentationExplorer(
    transforms=['positive_scale', 'bit_complement', 'mwis_penalty'],
    verify=True, seed=42,
)
exp_result = explorer.explore(h_problem, mwis_info=mwis_info)
print(f'Generated {len(exp_result.candidates)} Hamiltonian representations:')
for c in exp_result.candidates:
    rz = c.realizability
    lm = c.landscape
    gs = f'gap={lm.classical_gap:.3f}' if lm else ''
    print(f'  {c.id:20s} eq={c.equivalence_class.value:28s} '
          f'native={rz.native_pair_interactions} {gs}')

print('\nEquivalence verification:')
for c in exp_result.candidates:
    if c.verification is not None:
        v = c.verification
        print(f'  {c.id}: {v.classification.value} max_err={v.max_energy_error:.2e}')

## 3. Project A — Representation Scheduler

For a fixed Hamiltonian, search over QoolQit embeddings using successive
halving to avoid expensive emulation of all candidates.

In [ ]:
from qoolqit import AnalogDeviceWithDMM

scheduler = RepresentationScheduler(
    embedders=['interaction', 'spring', 'blade'], budget=12, seed=42, num_shots=200,
    halving=HalvingConfig(stage0_keep=1.0, stage2_keep=6, stage3_keep=3, stage4_keep=2),
)
sched_result = scheduler.search(
    h_problem, device=AnalogDeviceWithDMM(),
    run_emulation=True, run_robustness=True, robustness_samples=3,
)
print('Scheduler results:')
for s in sched_result.scores:
    if s.compilation_success and s.ground_state_probability is not None:
        print(f'  {s.candidate_id:20s} frob={s.frobenius_error:.4f} p_opt={s.ground_state_probability:.4f}')
best = sched_result.best('ground_state_probability')
if best:
    print(f'Best: {best.candidate_id} (p_opt={best.ground_state_probability:.4f})')

## 4. Combined Hierarchical Search — Factorial Experiment

Flagship experiment: Hamiltonian representations (varying penalty U) x
embedding candidates. Question: how much variation comes from H vs R?

In [ ]:
combined = combined_search(
    h_problem, mwis_info=mwis_info,
    n_hamiltonians=3, n_embedders=3, num_shots=200, seed=42,
    run_emulation=True, run_robustness=True, robustness_samples=3,
)
mat = combined.matrix('ground_state_probability')
print('Solution probability matrix (H x R):')
print(mat)
vd = combined.variance_decomposition('ground_state_probability')
print(f'\nη²(H)  — Hamiltonian main effect:  {vd["eta2_H"]:.1%}')
print(f'η²(R)  — Embedding main effect:    {vd["eta2_R"]:.1%}')
print(f'η²(H×R) — Interaction effect:       {vd["eta2_HxR"]:.1%}')
print(f'η²(res) — Residual:                 {vd["eta2_residual"]:.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(mat, cmap='viridis', aspect='auto')
ax.set_xticks(range(mat.shape[1])); ax.set_yticks(range(mat.shape[0]))
ax.set_xticklabels([f'R{i+1}' for i in range(mat.shape[1])])
ax.set_yticklabels(combined.hamiltonian_ids)
ax.set_xlabel('Embedding candidate'); ax.set_ylabel('Hamiltonian representation')
ax.set_title('Solution probability: H x R factorial')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.3f}', ha='center', va='center', color='w')
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 5. Key Finding

The flagship experiment (6 problems × 5 H × 9 R × 3 replicates = 810 cells)
shows that the **H×R interaction** is the dominant effect (median η² = 48.2%),
directly motivating joint representation search. Neither the Hamiltonian alone
nor the embedding alone determines performance — it is their *interaction*
that matters.

Source: `results/runs/flagship_v3/processed/report_metrics.json`

| Effect | η² (median) |
|--------|-------------|
| H (Hamiltonian) | 6.8% |
| R (Embedding) | 19.0% |
| H×R (Interaction) | 48.2% |

For 5 of 6 problems, the preselected fixed baseline fails terminal
ground-state preservation, while representation search identifies
physically valid alternatives for all six problems. The search enables
solutions that the default representation cannot reach at all.
QoolQit serves as the evaluation engine; WestQuant AI is optional.

In [ ]:
print('=== WestQuant Representation Stack — Complete ===')
print(f'QoolQit version: {qoolqit.__version__}')
print(f'Hamiltonians explored: {len(exp_result.candidates)}')
print(f'Factorial cells: {len(combined.cells)}')
print(f'η²(H):   {vd["eta2_H"]:.1%}')
print(f'η²(R):   {vd["eta2_R"]:.1%}')
print(f'η²(H×R): {vd["eta2_HxR"]:.1%}')